# SVD of flow past a cylinder

We analyze a data matrix of fluid flow past a cylinder, where each column is a flow field reshaped into a vector. The goals are:

1. compute the SVD of the flow data and interpret the matrices $U$, $\Sigma$, and $V^T$,
2. reconstruct the flow for different truncation ranks and compare the fidelity,
3. compute energy-based truncation ranks,
4. verify the interpretation of the coordinates $w_k$ in the reduced space,
5. build a linear regression model for the reduced amplitudes,
6. use that model to forecast future reduced states and flow snapshots.


## Learning goals

By the end of this notebook, you should be able to:

- compute the economy SVD of a flow snapshot matrix,
- interpret left singular vectors as dominant spatial flow structures,
- interpret $\Sigma V^T$ as time-dependent amplitudes,
- choose truncation ranks based on cumulative energy,
- measure reconstruction error with the Frobenius norm,
- verify the representation $x_k \approx \widetilde U w_k$,
- fit a linear dynamical model in reduced coordinates.


## 1. Load the reduced cylinder-flow data

We use a dataset stored as `data/cylinder_vorticity_reduced.npz`.

This file contains only what the notebook needs:

- `X`: the vorticity snapshot matrix,
- `nx`: number of grid points in the horizontal direction,
- `ny`: number of grid points in the vertical direction.


In [1]:
import numpy as np
import matplotlib.pyplot as plt

def load_cylinder_data():
    data = np.load("data/cylinder_vorticity.npz")
    X = np.asarray(data["X"], dtype=np.float32)
    nx = int(np.array(data["nx"]).squeeze())
    ny = int(np.array(data["ny"]).squeeze())

    return X, nx, ny

def plot_snapshot(vec, nx, ny, title="Snapshot", cmap="seismic"):
    plt.figure(figsize=(5, 4))
    field = vec.reshape(ny, nx)
    vmax = np.max(np.abs(field))
    plt.imshow(field, cmap=cmap, origin="lower", aspect="auto", vmin=-vmax, vmax=vmax)
    plt.colorbar()
    plt.title(title)
    plt.tight_layout()
    plt.show()
    

In [2]:
X, nx, ny = load_cylinder_data()

print("Snapshot matrix shape:", X.shape)
print("nx =", nx, "ny =", ny)


Snapshot matrix shape: (89351, 151)
nx = 199 ny = 449


## 2. Visualize a few flow snapshots

We first inspect several columns of the data matrix to understand what one snapshot looks like and how the vorticity field evolves in time.

Since `nx` and `ny` are provided by the dataset, each snapshot can be reshaped into a two-dimensional field.


In [ ]:
# TODO: choose a few snapshot indices and plot them

# indices = [0, X.shape[1] // 4, X.shape[1] // 2, 3 * X.shape[1] // 4]

# for k in indices:
#     plot_snapshot(X[:, k], nx=nx, ny=ny, title=f"Snapshot {k}")


## 3. Compute the SVD of the flow data

We compute the economy SVD
$$
X = U\Sigma V^T.
$$

Here:
- the columns of $U$ are dominant spatial flow structures,
- the singular values in $\Sigma$ measure their importance,
- the rows of $\Sigma V^T$ contain the time-dependent amplitudes.


In [ ]:
# TODO: compute the economy SVD of X

# U, s, VT = ...
# Sigma = np.diag(s)

# print(U.shape, s.shape, VT.shape)


## 4. Plot the singular value spectrum and leading modes

We now visualize:
- the singular value spectrum,
- the cumulative energy,
- the first few left singular vectors.

If `nx` and `ny` are known, the left singular vectors are reshaped into spatial flow fields.


In [ ]:
# TODO: plot the singular values on a semilog scale
# TODO: plot the cumulative energy based on the squared singular values


In [ ]:
# TODO: plot the first 4 left singular vectors
# If nx and ny are known, reshape U[:, j] to a 2D field.


## 5. Reconstruct the flow with different truncation ranks

We reconstruct the snapshot matrix using truncated SVD:
$$
\widetilde X_r = U_r \Sigma_r V_r^T.
$$

We use ranks chosen to capture:
- 90% of the total energy,
- 99% of the total energy,
- 99.9% of the total energy.

Recall that the energy is based on the squared singular values.


In [ ]:
# TODO: compute the ranks that capture 90%, 99%, and 99.9% of the total energy

# energy = ...
# cumulative_energy = ...

# r90 = ...
# r99 = ...
# r999 = ...

# print(r90, r99, r999)


In [ ]:
# TODO: reconstruct X with the three truncation ranks

# def truncated_svd_reconstruction(U, s, VT, r):
#     return ...

# X90 = ...
# X99 = ...
# X999 = ...


In [ ]:
# TODO: compute the squared Frobenius norm errors

# err90 = ...
# err99 = ...
# err999 = ...

# print(err90, err99, err999)


In [ ]:
# TODO: compare one snapshot of X with the three truncated reconstructions

# k = X.shape[1] // 2

# plot_snapshot(X[:, k], nx=nx, ny=ny, title="Original snapshot")
# plot_snapshot(X90[:, k], nx=nx, ny=ny, title=f"90% energy (r={r90})")
# plot_snapshot(X99[:, k], nx=nx, ny=ny, title=f"99% energy (r={r99})")
# plot_snapshot(X999[:, k], nx=nx, ny=ny, title=f"99.9% energy (r={r999})")

#To save memory
del X90, X999
import gc
gc.collect()

## 6. Fix $r=10$ and interpret the reduced coordinates

Let
$$
\widetilde X = \widetilde U \widetilde \Sigma \widetilde V^T,
$$
with $r=10$, and define
$$
W = \widetilde \Sigma \widetilde V^T.
$$

Each column $w_k \in \mathbb{R}^{10}$ contains the amplitudes of the first 10 eigenflows in snapshot $k$.
We verify that
$$
x_k \approx \widetilde U w_k.
$$


In [ ]:
r = 10

# TODO: build U_tilde, Sigma_tilde, VT_tilde, and W

# U_tilde = ...
# Sigma_tilde = ...
# VT_tilde = ...
# W = ...


In [ ]:
# TODO: choose one snapshot index k and compare x_k with U_tilde @ w_k

# k = X.shape[1] // 3
# xk = X[:, k]
# wk = W[:, k]
# xk_recon = ...

# print("Relative reconstruction error:", ...)


## 7. Fit a linear dynamical model in reduced coordinates

We now build the regression model
$$
w_{k+1} = A w_k.
$$

If we define
$$
W_1 = [w_1, w_2, \dots, w_{m-1}], \qquad
W_2 = [w_2, w_3, \dots, w_m],
$$
then the least-squares matrix is
$$
A = W_2 W_1^\dagger.
$$


In [ ]:
# TODO: build W1 and W2, then compute the least-squares matrix A_red

# W1 = ...
# W2 = ...
# A_red = ...

# print(A_red.shape)


In [ ]:
# TODO: compute one-step prediction errors in reduced coordinates

# W2_pred = ...
# rel_err_reduced = ...

# print(rel_err_reduced)


In [ ]:
# TODO: start from an initial reduced state and forecast several future steps

# n_forecast = 20
# w_forecast = ...
# for j in range(...):
#     ...

# Compare one predicted snapshot with the true snapshot.


## 8. Animate the original and reconstructed flows

A movie often gives a better sense of the coherent structures in the wake than static snapshots.

In this section, we build:

- one comparison movie with **original** and **reconstructed** flow shown side by side,
- one separate movie for the **reconstruction error**.

This way, the error can use its own color scale and becomes much easier to see.


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def animate_two_flow_comparison(X_left, X_right, nx, ny,
                                title_left="Original",
                                title_right="Reconstructed",
                                cmap="seismic", interval=50):
    """
    Return a JavaScript animation with two side-by-side subplots
    sharing the same color scale.
    """
    n_frames = min(X_left.shape[1], X_right.shape[1])

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    vmax = max(
        np.max(np.abs(X_left)),
        np.max(np.abs(X_right)),
    )

    mats = [X_left, X_right]
    titles = [title_left, title_right]
    ims = []

    for ax, Xmat, ttl in zip(axes, mats, titles):
        field0 = Xmat[:, 0].reshape(ny, nx)
        im = ax.imshow(
            field0,
            cmap=cmap,
            origin="lower",
            aspect="auto",
            vmin=-vmax,
            vmax=vmax,
        )
        ax.set_title(f"{ttl}: frame 0")
        ax.axis("off")
        ims.append(im)

    fig.colorbar(ims[-1], ax=axes.ravel().tolist(), shrink=0.8)

    def update(frame):
        for ax, im, Xmat, ttl in zip(axes, ims, mats, titles):
            field = Xmat[:, frame].reshape(ny, nx)
            im.set_array(field)
            ax.set_title(f"{ttl}: frame {frame}")
        return ims

    anim = FuncAnimation(fig, update, frames=n_frames, interval=interval, blit=True)
    plt.close(fig)
    return anim

def animate_error_flow(X_err, nx, ny, title="Error", cmap="seismic", interval=50):
    """
    Return a JavaScript animation for the error field alone, using its own color scale.
    """
    n_frames = X_err.shape[1]

    fig, ax = plt.subplots(figsize=(5.5, 4))

    vmax = np.max(np.abs(X_err))
    field0 = X_err[:, 0].reshape(ny, nx)

    im = ax.imshow(
        field0,
        cmap=cmap,
        origin="lower",
        aspect="auto",
        vmin=-vmax,
        vmax=vmax,
    )
    ax.set_title(f"{title}: frame 0")
    ax.axis("off")
    fig.colorbar(im, ax=ax, shrink=0.8)

    def update(frame):
        field = X_err[:, frame].reshape(ny, nx)
        im.set_array(field)
        ax.set_title(f"{title}: frame {frame}")
        return [im]

    anim = FuncAnimation(fig, update, frames=n_frames, interval=interval, blit=True)
    plt.close(fig)
    return anim


Choose one of your reconstructed matrices, for example `X99` or `X999`, and compare it with the original flow. Then animate the error field separately.


In [ ]:
# TODO: choose a reconstructed flow matrix, for example:
# X_recon = X99

# TODO: build the error matrix
# X_err = X - X_recon


In [ ]:
# TODO: create the movies

# anim_compare = animate_two_flow_comparison(
#     X, X_recon, nx, ny,
#     title_left="Original flow",
#     title_right="Reconstructed flow",
# )

# anim_error = animate_error_flow(
#     X_err, nx, ny,
#     title="Reconstruction error",
# )


In [ ]:
# TODO: display the comparison movie
# HTML(anim_compare.to_jshtml())


In [ ]:
# TODO: display the error movie
# HTML(anim_error.to_jshtml())


## 9. Animate the predicted flow from the reduced linear model

We now use the reduced linear regression model
$$
w_{k+1} = A w_k
$$
to generate predicted reduced states, map them back to the full flow space, and build:

- one comparison movie with **true** and **predicted** flow shown side by side,
- one separate movie for the **prediction error**.

Again, the separate error movie makes the smaller-scale error structures easier to visualize.


In [ ]:
# TODO: choose how many frames to predict
# n_pred = 30

# TODO: build a matrix of predicted reduced states
# W_pred = np.zeros((r, n_pred))
# W_pred[:, 0] = W[:, 0]
# for j in range(n_pred - 1):
#     W_pred[:, j + 1] = A_red @ W_pred[:, j]


In [ ]:
# TODO: map the predicted reduced states back to the full flow space
# X_pred = U_tilde @ W_pred

# TODO: extract the matching true snapshots for comparison
# X_true_pred = X[:, :n_pred]

# TODO: build the prediction error matrix
# X_pred_err = X_true_pred - X_pred


In [ ]:
# TODO: create the movies

# anim_pred_compare = animate_two_flow_comparison(
#     X_true_pred, X_pred, nx, ny,
#     title_left="True flow",
#     title_right="Predicted flow",
# )

# anim_pred_error = animate_error_flow(
#     X_pred_err, nx, ny,
#     title="Prediction error",
# )


In [ ]:
# TODO: display true vs predicted
# HTML(anim_pred_compare.to_jshtml())


In [ ]:
# TODO: display prediction error
# HTML(anim_pred_error.to_jshtml())


## Final discussion

1. What do the leading singular vectors represent in this fluid-flow dataset?
2. How many modes are needed to capture 90%, 99%, and 99.9% of the total energy?
3. Why does the approximation improve as the truncation rank increases?
4. What is the meaning of the reduced coordinates $w_k$?
5. Why is it reasonable to model the reduced coordinates with a linear system?
6. Where do you expect the regression model to perform well, and where might it fail?
